# Mini-GPT on Google Colab (GPU)

Train the decoder-only Transformer from the **Mini-GPT** repo on a free Colab GPU.

**Before running:** set the runtime to GPU via **Runtime -> Change runtime type -> Hardware accelerator: GPU**, then **Runtime -> Run all**.

The notebook clones the repo, installs PyTorch (already present on Colab), trains the model, and generates a sample of pseudo-Shakespeare.

## 1. Check the GPU
Confirm a GPU is attached. If this errors or shows nothing, set the runtime to GPU first.

In [ ]:
!nvidia-smi

## 2. Clone the repo
Pulls the exact code (data pipeline, attention, blocks, model, training loop) from GitHub.

In [ ]:
!git clone https://github.com/shubham-KMX/Mini-GPT.git
%cd Mini-GPT

## 3. Verify PyTorch sees CUDA
Colab ships with PyTorch preinstalled, so no `pip install` is normally needed.

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Train
On a GPU we can afford a larger model than the CPU defaults. `train.py` auto-detects CUDA and moves the model and batches onto it.

This config (~256-dim, 6 layers, 6 heads, context 256) trains in a few minutes on a T4 and reaches a val loss around 1.5, which produces convincing pseudo-Shakespeare. Adjust the flags to taste.

In [ ]:
!python train.py \
  --batch-size 64 \
  --block-size 256 \
  --n-embd 256 \
  --n-head 6 \
  --n-layer 6 \
  --dropout 0.2 \
  --learning-rate 3e-4 \
  --max-iters 5000 \
  --eval-interval 500 \
  --gen-tokens 1000

## 5. Generate more text from the saved checkpoint
`train.py` saves `mini_gpt.pt`. Reload it and sample without retraining.

In [ ]:
import torch
from model import GPTLanguageModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('mini_gpt.pt', map_location=device)
cfg = ckpt['config']
chars = ckpt['vocab']

# Rebuild the tokenizer mappings from the saved vocabulary
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
decode = lambda ids: ''.join(itos[i] for i in ids)
encode = lambda s: [stoi[c] for c in s]

model = GPTLanguageModel(
    vocab_size=len(chars),
    n_embd=cfg['n_embd'],
    block_size=cfg['block_size'],
    n_layer=cfg['n_layer'],
    n_head=cfg['n_head'],
    dropout=cfg['dropout'],
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Prompt the model with some text (or start from a newline)
prompt = 'ROMEO:'
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = model.generate(context, max_new_tokens=1000)
print(decode(out[0].tolist()))

## 6. (Optional) Download the checkpoint
Save the trained weights to your machine.

In [ ]:
from google.colab import files
files.download('mini_gpt.pt')